# MILK10k — Step 5: Conformal prediction sets and a guaranteed triage rule
Split-conformal, cross-fitted: for test fold *f*, the calibration set is the out-of-fold predictions of the other 4 folds.
* Coverage per availability pattern (marginal, per class, per skin-tone group).
* **Two-stage triage rule** (reassure / refer / request dermoscopy) with a class-conditional guarantee on missed treatable lesions — see below.

In [ ]:
import os, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
from scipy.optimize import minimize_scalar

def find_file(name, tops=("/kaggle/input", "/kaggle/working")):
    for top in tops:
        for root, dirs, files in os.walk(top, followlinks=True):
            if name in files: return os.path.join(root, name)
    return None
NPZ = find_file("oof_probs_fusion.npz"); FOLDS_CSV = find_file("milk10k_lesions_folds.csv")
assert NPZ and FOLDS_CSV, "attach notebook-o3 and notebook-o2 outputs"
OUT = "/kaggle/working/stage5"; os.makedirs(OUT, exist_ok=True)
les = pd.read_csv(FOLDS_CSV)
CLASSES = ['AKIEC','BCC','BEN_OTH','BKL','DF','INF','MAL_OTH','MEL','NV','SCCKA','VASC']; K = len(CLASSES)
ACTIONABLE = ["AKIEC", "BCC", "MAL_OTH", "MEL", "SCCKA"]; is_act_cls = np.isin(CLASSES, ACTIONABLE)
y = les["y"].values.astype(int); fold = les["fold"].values; n = len(y)
y_act = is_act_cls[y]
tone = pd.cut(les["skin_tone_class"], [-1, 2, 3, 5], labels=["darker(0-2)", "medium(3)", "lighter(4-5)"]).astype(str).values
PATTERNS = ["CDM", "CD", "CM", "DM", "C", "D", "M"]
npz = np.load(NPZ); eps = 1e-9

# cross-fitted temperature scaling (same as 04b)
def temp(P, T):
    L = np.log(P + eps) / T; L -= L.max(1, keepdims=True); E = np.exp(L); return E / E.sum(1, keepdims=True)
def nll(P, yy): return -np.log(P[np.arange(len(yy)), yy] + eps).mean()
PROB = {}
for p in PATTERNS:
    P = npz[f"late_drop|{p}"].astype(float); out = np.zeros_like(P)
    for f in range(5):
        tr, te = fold != f, fold == f
        T = minimize_scalar(lambda t: nll(temp(P[tr], t), y[tr]), bounds=(0.2, 5), method="bounded").x
        out[te] = temp(P[te], T)
    PROB[p] = out
print("patterns:", list(PROB))

## Conformal sets (LAC score = 1 − p(true class)); marginal and class-conditional (Mondrian)

In [ ]:
def conformal_sets(P, alpha, mondrian=False):
    """Cross-fitted split conformal. Returns boolean (n, K) prediction sets."""
    S = np.zeros((n, K), bool)
    score_true = 1 - P[np.arange(n), y]
    for f in range(5):
        cal, te = fold != f, fold == f
        if not mondrian:
            s = np.sort(score_true[cal]); m = len(s)
            qhat = s[min(m - 1, int(np.ceil((m + 1) * (1 - alpha))) - 1)]
            S[te] = (1 - P[te]) <= qhat
        else:
            for k in range(K):
                s = np.sort(score_true[cal & (y == k)]); m = len(s)
                qhat = 1.0 if m == 0 or np.ceil((m + 1) * (1 - alpha)) > m else s[int(np.ceil((m + 1) * (1 - alpha))) - 1]
                S[te, k] = (1 - P[te, k]) <= qhat
    return S

def coverage_table(alpha, mondrian):
    rows = []
    for p in PATTERNS:
        S = conformal_sets(PROB[p], alpha, mondrian); cov = S[np.arange(n), y]
        cls_cov = [cov[y == k].mean() for k in range(K)]
        rows.append(dict(pattern=p, coverage=cov.mean(), mean_set_size=S.sum(1).mean(),
                         singleton_rate=(S.sum(1) == 1).mean(), worst_class_cov=min(cls_cov),
                         worst_class=CLASSES[int(np.argmin(cls_cov))],
                         **{f"cov_{g}": cov[tone == g].mean() for g in ["darker(0-2)", "medium(3)", "lighter(4-5)"]}))
    return pd.DataFrame(rows).set_index("pattern")

for alpha in [0.10, 0.05]:
    for mond in [False, True]:
        t = coverage_table(alpha, mond); t.to_csv(f"{OUT}/coverage_a{alpha}_{'mondrian' if mond else 'marginal'}.csv")
        print(f"\n=== alpha={alpha} ({1-alpha:.0%} target), {'class-conditional' if mond else 'marginal'} ===")
        display(t.round(3))

## Triage rule: binary (treat vs benign) class-conditional conformal, two stages
The 11-class sets above are too wide to decide on (they almost always mix treatable and benign classes), so the decision uses a **two-class** conformal step on the actionable score s = Σ p(actionable classes):
* "actionable" is in the set if 1 − s ≤ q_A (q_A calibrated on actionable lesions, level α_A)
* "benign" is in the set if s ≤ q_B (q_B calibrated on benign lesions, level α_B)
* set {benign} → **reassure**, {actionable} → **refer**, both/empty → **request dermoscopy** (stage 1, clinical + metadata); stage 2 repeats this with the all-inputs model, **calibrated only on lesions that stage 1 sent for dermoscopy**; still ambiguous → refer.
* Guarantee (class-conditional split conformal): **P(actionable lesion reassured) ≤ α_A(stage 1) + α_A(stage 2)**. We split the miss budget α equally over the two stages.
* α_B trades benign reassurance against dermoscopy/referral load; we sweep it.

In [ ]:
s_act = {p: PROB[p][:, is_act_cls].sum(1) for p in ["CM", "CDM"]}
def q_upper(scores, a):
    s = np.sort(scores); m = len(s); k = int(np.ceil((m + 1) * (1 - a)))
    return np.inf if k > m else s[k - 1]

def binary_sets(s, cal_mask_fn, aA, aB):
    """Cross-fitted two-class Mondrian conformal. cal_mask_fn(f) -> boolean calibration mask for test fold f."""
    inA, inB = np.zeros(n, bool), np.zeros(n, bool)
    for f in range(5):
        te, cal = fold == f, cal_mask_fn(f)
        qA = q_upper(1 - s[cal & y_act], aA); qB = q_upper(s[cal & ~y_act], aB)
        inA[te] = (1 - s[te]) <= qA; inB[te] = s[te] <= qB
    return inA, inB

def decide(inA, inB): return np.where(inA & ~inB, "refer", np.where(inB & ~inA, "reassure", "ambiguous"))

def run(alpha, aB, strategy="adaptive"):
    aA = alpha / 2
    if strategy == "all":                      # dermoscopy for everyone, single stage, full budget
        inA, inB = binary_sets(s_act["CDM"], lambda f: fold != f, alpha, aB)
        d = decide(inA, inB); return np.where(d == "ambiguous", "refer", d), np.ones(n, bool)
    inA, inB = binary_sets(s_act["CM"], lambda f: fold != f, aA if strategy == "adaptive" else alpha, aB)
    d1 = decide(inA, inB)
    if strategy == "none":
        return np.where(d1 == "ambiguous", "refer", d1), np.zeros(n, bool)
    req = d1 == "ambiguous"
    inA2, inB2 = binary_sets(s_act["CDM"], lambda f: (fold != f) & req, aA, aB)   # calibrated on stage-1 ambiguous
    d2 = decide(inA2, inB2)
    final = d1.copy(); final[req] = np.where(d2[req] == "ambiguous", "refer", d2[req])
    return final, req

def summarise(final, req, **tags):
    reassured, referred = final == "reassure", final == "refer"
    return {**tags, "dermoscopy_rate": req.mean(), "referral_rate": referred.mean(),
            "act_missed": (reassured & y_act).sum() / y_act.sum(),
            "benign_reassured": (reassured & ~y_act).sum() / (~y_act).sum(),
            "MEL_missed": (reassured & (y == 7)).sum() / (y == 7).sum(),
            "SCCKA_missed": (reassured & (y == 9)).sum() / (y == 9).sum(),
            "BCC_missed": (reassured & (y == 1)).sum() / (y == 1).sum()}

rows = []
for alpha in [0.05, 0.10]:
    for aB in [0.10, 0.20, 0.30]:
        for strat in ["none", "adaptive", "all"]:
            final, req = run(alpha, aB, strat)
            rows.append(summarise(final, req, alpha=alpha, alpha_B=aB, strategy=strat))
tri = pd.DataFrame(rows); tri.to_csv(f"{OUT}/triage_rule.csv", index=False)
tri.set_index(["alpha", "alpha_B", "strategy"]).round(3)

## Guarantee check by skin-tone group (main setting: α = 0.10, α_B = 0.20)

In [ ]:
final, req = run(0.10, 0.20, "adaptive")
g = pd.DataFrame({"tone": tone, "act": y_act, "reassured": final == "reassure", "req": req, "ref": final == "refer"})
by_tone = g.groupby("tone").apply(lambda d: pd.Series({
    "n": len(d), "n_actionable": d["act"].sum(),
    "dermoscopy_rate": d["req"].mean(), "referral_rate": d["ref"].mean(),
    "act_missed": (d["reassured"] & d["act"]).sum() / max(d["act"].sum(), 1),
    "benign_reassured": (d["reassured"] & ~d["act"]).sum() / max((~d["act"]).sum(), 1)}))
by_tone.to_csv(f"{OUT}/triage_by_tone.csv"); by_tone.round(3)

## Who is sent for dermoscopy? (α = 0.10, α_B = 0.20)

In [ ]:
cls = pd.Series(y).map(dict(enumerate(CLASSES)))
who = pd.DataFrame({"n": cls.value_counts(), "sent": cls[req].value_counts(),
                    "reassured": cls[final == "reassure"].value_counts(), "referred": cls[final == "refer"].value_counts()}).fillna(0).astype(int)
who["sent_rate"] = (who["sent"] / who["n"]).round(3); who["actionable"] = [c in ACTIONABLE for c in who.index]
who.to_csv(f"{OUT}/triage_by_class.csv"); who.sort_values("sent_rate", ascending=False)